# Practical PyTorch · Part 11 — Companion Notebook

This goes with **"Logits, Softmax, and the Two Ends of Every Model"**. Run the cells top to bottom (Shift+Enter).

We'll take one real image through the whole journey: **preprocess → batch → forward → postprocess**, using a pretrained ResNet-50 from torchvision. No training, no math — just the two ends of a model.

## 1. Load a model and grab a picture

`ResNet50_Weights.DEFAULT` is a specific set of trained numbers. The weights object also carries the preprocessing and the label list — we'll use both. The first run downloads the weights, so give it a few seconds.

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()   # evaluation mode: switch off training-time behavior (do this once)

print("Model loaded. It knows", len(weights.meta["categories"]), "categories.")

In [ ]:
import urllib.request
from PIL import Image

# A friendly stock photo of a dog.
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
urllib.request.urlretrieve(url, "image.jpg")

img = Image.open("image.jpg")
img

## 2. The input side: preprocess

A model doesn't eat a JPEG — it eats a tensor of a specific shape and scale. The **correct preprocessing belongs to the model**, so we ask the weights for it instead of guessing the resize size and normalization numbers ourselves.

In [ ]:
preprocess = weights.transforms()   # the exact pipeline this model was trained with
print(preprocess)

tensor = preprocess(img)
print("shape:", tensor.shape)   # [3, 224, 224] = channels, height, width
print("dtype:", tensor.dtype)

### Add the batch dimension

Models expect a leading dimension that says *how many* images you're sending. We have one, so the batch size is 1 — but we still have to say so. `unsqueeze(0)` inserts a size-1 dimension at the front.

In [ ]:
batch = tensor.unsqueeze(0)
print("before:", tensor.shape)   # [3, 224, 224]
print("after: ", batch.shape)    # [1, 3, 224, 224] — a batch of 1 image

## 3. The forward pass

We're not training, so we skip gradient bookkeeping with `torch.inference_mode()` — faster and lighter. (You'll also see the older `torch.no_grad()`; same idea.)

In [ ]:
with torch.inference_mode():
    out = model(batch)

print("output shape:", out.shape)   # [1, 1000] — one score per known class
print("first 5 raw scores:", out[0][:5])

## 4. The output side: logits → probabilities → answer

Those raw scores are **logits**: higher means more likely, but they aren't probabilities — they don't sit in 0–1 and don't sum to anything meaningful. `softmax` fixes that.

In [ ]:
logits = out
probs = logits.softmax(dim=-1)   # dim=-1 = across the 1000 classes

print("do the probabilities sum to 1?", probs.sum().item())

`argmax` gives the *index* of the biggest probability — the winning class number. Then we look that index up in the model's label list to get a human-readable name.

In [ ]:
labels = weights.meta["categories"]

top = probs.argmax(dim=-1)
print("top class index:", top.item())
print("top label:      ", labels[top.item()])

### Top-k: the runners-up

A single answer hides how sure the model is. `torch.topk` returns the top *k* probabilities *and* their indices, so we can show what else it considered.

In [ ]:
values, indices = torch.topk(probs, k=5)

for score, idx in zip(values[0], indices[0]):
    print(f"{labels[idx]:<25} {score.item():.1%}")

## 5. The general recipe

Here's the whole journey in one cell. The details inside each step change from model to model — the four steps don't: **preprocess → batch → forward → postprocess.**

In [ ]:
def classify(pil_image, k=5):
    model.eval()                                  # 1. eval mode
    batch = preprocess(pil_image).unsqueeze(0)    # 2. preprocess + add batch dim
    with torch.inference_mode():                  # 3. forward, no gradients
        logits = model(batch)
    probs = logits.softmax(dim=-1)                # 4. postprocess
    values, indices = torch.topk(probs, k=k)
    return [(labels[i], v.item()) for v, i in zip(values[0], indices[0])]

for name, score in classify(img):
    print(f"{name:<25} {score:.1%}")

## 6. Try it on your own image

Swap in any URL — the recipe doesn't care what the picture is. Try something the model probably knows (a cat, a car, a coffee mug) and watch the top-5 shift.

In [ ]:
your_url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"  # <- change me
urllib.request.urlretrieve(your_url, "yours.jpg")
your_img = Image.open("yours.jpg").convert("RGB")

for name, score in classify(your_img):
    print(f"{name:<25} {score:.1%}")
your_img

That's both ends of a model demystified. Next up (Part 16): how text becomes model input — tokens, attention masks, padding, and batching — the bridge that makes `pipeline()` make sense.